In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

dataset_path = "/content/drive/MyDrive/Drowsiness"

print(os.listdir(dataset_path))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Drowsiness'

In [5]:
import zipfile
import os

extract_path = "/content/uta_rldd_fold1"
os.makedirs(extract_path, exist_ok=True)

zip_files = [
    "Fold1_part1.zip",
    "Fold1_part2.zip"
]

for zip_name in zip_files:
    zip_path = os.path.join(dataset_path, zip_name)
    print("Extracting:", zip_name)

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

print("Done.")
print(os.listdir(extract_path))

Extracting: Fold1_part1.zip


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Drowsiness/Fold1_part1.zip'

In [ ]:
import os

base_path = "/content/uta_rldd_fold1"

for folder in os.listdir(base_path):
    folder_path = os.path.join(base_path, folder)
    print("\nMAIN:", folder)
    print(os.listdir(folder_path)[:20])


MAIN: Fold1_part1
['02', '04', '05', '06', '03', '01']

MAIN: Fold1_part2
['10', '11', '07', '09', '08', '12']


In [ ]:
import os

base_path = "/content/uta_rldd_fold1"

for main_folder in os.listdir(base_path):
    main_path = os.path.join(base_path, main_folder)

    for subject in os.listdir(main_path):
        subject_path = os.path.join(main_path, subject)

        if os.path.isdir(subject_path):
            print("\nSUBJECT:", subject_path)
            print(os.listdir(subject_path)[:20])


SUBJECT: /content/uta_rldd_fold1/Fold1_part1/02
['10.MOV', '0.mov', '5.MOV']

SUBJECT: /content/uta_rldd_fold1/Fold1_part1/04
['10.mp4', '5.mp4', '0.mp4']

SUBJECT: /content/uta_rldd_fold1/Fold1_part1/05
['0.MOV', '10.MOV', '5.MOV']

SUBJECT: /content/uta_rldd_fold1/Fold1_part1/06
['10.mp4', '5.mp4', '0.mp4']

SUBJECT: /content/uta_rldd_fold1/Fold1_part1/03
['0.MOV', '10.mov', '5.mov']

SUBJECT: /content/uta_rldd_fold1/Fold1_part1/01
['10.MOV', '0.mov', '5.mov']

SUBJECT: /content/uta_rldd_fold1/Fold1_part2/10
['0.MOV', '10.MOV', '5.MOV']

SUBJECT: /content/uta_rldd_fold1/Fold1_part2/11
['10.mp4', '5.mp4', '0.mp4']

SUBJECT: /content/uta_rldd_fold1/Fold1_part2/07
['10.mp4', '5.mp4', '0.mp4']

SUBJECT: /content/uta_rldd_fold1/Fold1_part2/09
['10.mp4', '5.mp4', '0.mp4']

SUBJECT: /content/uta_rldd_fold1/Fold1_part2/08
['10.mp4', '5.mp4', '0.mp4']

SUBJECT: /content/uta_rldd_fold1/Fold1_part2/12
['10.mp4', '5.mp4', '0.mp4']


In [ ]:
import os
import cv2
import numpy as np

BASE_PATH = "/content/uta_rldd_fold1"
IMG_SIZE = 64
SEQUENCE_LENGTH = 10
FRAME_STEP = 10
MAX_FRAMES_PER_VIDEO = 300

LABEL_MAP = {
    "0": 0,    # Alert
    "10": 1   # Drowsy
}

sequences = []
labels = []
video_count = {"alert": 0, "drowsy": 0}
frame_count = {"alert": 0, "drowsy": 0}

def get_class_from_filename(filename):
    name = filename.lower()
    if name.startswith("0."):
        return "0"
    elif name.startswith("10."):
        return "10"
    return None

for fold_part in os.listdir(BASE_PATH):
    fold_path = os.path.join(BASE_PATH, fold_part)

    for subject in os.listdir(fold_path):
        subject_path = os.path.join(fold_path, subject)

        if not os.path.isdir(subject_path):
            continue

        for video_file in os.listdir(subject_path):
            class_key = get_class_from_filename(video_file)

            if class_key is None:
                continue

            video_path = os.path.join(subject_path, video_file)
            label = LABEL_MAP[class_key]
            class_name = "alert" if label == 0 else "drowsy"

            print("Processing:", video_path, "Label:", class_name)

            cap = cv2.VideoCapture(video_path)
            frames = []
            frame_index = 0
            saved_frames = 0

            while True:
                ret, frame = cap.read()

                if not ret:
                    break

                if frame_index % FRAME_STEP == 0:
                    frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    frame = frame / 255.0
                    frames.append(frame)

                    frame_count[class_name] += 1
                    saved_frames += 1

                frame_index += 1

                if saved_frames >= MAX_FRAMES_PER_VIDEO:
                    break

            cap.release()

            for i in range(0, len(frames) - SEQUENCE_LENGTH + 1, SEQUENCE_LENGTH):
                sequences.append(frames[i:i + SEQUENCE_LENGTH])
                labels.append(label)

            video_count[class_name] += 1

sequences = np.array(sequences, dtype=np.float32)
labels = np.array(labels, dtype=np.int64)

print("Sequences shape:", sequences.shape)
print("Labels shape:", labels.shape)
print("Video count:", video_count)
print("Frame count:", frame_count)

Processing: /content/uta_rldd_fold1/Fold1_part1/02/10.MOV Label: drowsy
Processing: /content/uta_rldd_fold1/Fold1_part1/02/0.mov Label: alert
Processing: /content/uta_rldd_fold1/Fold1_part1/04/10.mp4 Label: drowsy
Processing: /content/uta_rldd_fold1/Fold1_part1/04/0.mp4 Label: alert
Processing: /content/uta_rldd_fold1/Fold1_part1/05/0.MOV Label: alert
Processing: /content/uta_rldd_fold1/Fold1_part1/05/10.MOV Label: drowsy
Processing: /content/uta_rldd_fold1/Fold1_part1/06/10.mp4 Label: drowsy
Processing: /content/uta_rldd_fold1/Fold1_part1/06/0.mp4 Label: alert
Processing: /content/uta_rldd_fold1/Fold1_part1/03/0.MOV Label: alert
Processing: /content/uta_rldd_fold1/Fold1_part1/03/10.mov Label: drowsy
Processing: /content/uta_rldd_fold1/Fold1_part1/01/10.MOV Label: drowsy
Processing: /content/uta_rldd_fold1/Fold1_part1/01/0.mov Label: alert
Processing: /content/uta_rldd_fold1/Fold1_part2/10/0.MOV Label: alert
Processing: /content/uta_rldd_fold1/Fold1_part2/10/10.MOV Label: drowsy
Proces

In [ ]:
import os
import numpy as np

save_path = "/content/drive/MyDrive/drowsiness_project/processed_data"
os.makedirs(save_path, exist_ok=True)

np.save(os.path.join(save_path, "sequences.npy"), sequences)
np.save(os.path.join(save_path, "labels.npy"), labels)

print("Saved to:", save_path)

Saved to: /content/drive/MyDrive/drowsiness_project/processed_data


In [ ]:
from sklearn.model_selection import train_test_split
import os
import numpy as np

X_train, X_temp, y_train, y_temp = train_test_split(
    sequences, labels, test_size=0.30, random_state=42, stratify=labels
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

np.save(os.path.join(save_path, "X_train.npy"), X_train)
np.save(os.path.join(save_path, "y_train.npy"), y_train)
np.save(os.path.join(save_path, "X_val.npy"), X_val)
np.save(os.path.join(save_path, "y_val.npy"), y_val)
np.save(os.path.join(save_path, "X_test.npy"), X_test)
np.save(os.path.join(save_path, "y_test.npy"), y_test)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (504, 10, 64, 64, 3) (504,)
Validation: (108, 10, 64, 64, 3) (108,)
Test: (108, 10, 64, 64, 3) (108,)
